In [ ]:
%pip install -q polars plotly gdown python-dotenv kailash-align


# ════════════════════════════════════════════════════════════════════════
# ASCENT09 — Exercise 1: LoRA Internals and Rank Selection
# ════════════════════════════════════════════════════════════════════════
# OBJECTIVE: Derive LoRA from first principles — implement low-rank
#   decomposition, LoRA forward pass, and rank selection heuristics.
#   Validate from-scratch implementations against AlignmentPipeline.
#
# TASKS:
#   1. Implement low-rank matrix decomposition (SVD-based)
#   2. Measure reconstruction error vs rank vs parameter count
#   3. Implement LoRA forward pass from scratch
#   4. Use AlignmentPipeline with LoRAConfig at multiple ranks
#   5. Implement rank selector heuristic (stable rank)
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations

import os

import numpy as np
import polars as pl

from kailash_align import AlignmentConfig, AlignmentPipeline, AdapterRegistry
from kailash_align.config import SFTConfig, LoRAConfig
from kailash_align.registry import AdapterSignature

from shared import ASCENTDataLoader
from shared.kailash_helpers import setup_environment

setup_environment()

base_model = os.environ.get("SFT_BASE_MODEL", "TinyLlama/TinyLlama-1.1B-Chat-v1.0")



### Data Loading


In [ ]:
loader = ASCENTDataLoader()
reports = loader.load("ascent09", "sg_company_reports.parquet")

print(f"=== SG Company Reports Dataset ===")
print(f"Shape: {reports.shape}")
print(f"Columns: {reports.columns}")
print(reports.head(3))



## TASK 1: Implement low-rank matrix decomposition from scratch (SVD)


In [ ]:
# Full fine-tuning updates W directly: W' = W + delta_W
# LoRA insight: delta_W is low-rank during fine-tuning.
# Factor delta_W = B @ A where B in R^{d x r}, A in R^{r x k}, r << min(d, k)

rng = np.random.default_rng(42)

# TODO: Create d_model=512, d_input=512, pretrained weight matrix, and delta_W (* 0.01)
d_model = ____
d_input = ____
W_pretrained = ____
delta_W = ____


def low_rank_decomposition(
    matrix: np.ndarray, rank: int
) -> tuple[np.ndarray, np.ndarray]:
    """Decompose matrix into B @ A using truncated SVD.

    Args:
        matrix: (d, k) matrix to decompose
        rank: target rank r

    Returns:
        B: (d, r) matrix, A: (r, k) matrix such that matrix ≈ B @ A
    """
    # TODO: Compute SVD: U, S, Vt = np.linalg.svd(matrix, full_matrices=False)
    # Truncate to rank r, split singular values between B and A:
    #   sqrt_S = np.sqrt(S[:rank])
    #   B = U[:, :rank] * sqrt_S[np.newaxis, :]
    #   A = sqrt_S[:, np.newaxis] * Vt[:rank, :]
    ____
    ____
    ____
    ____
    ____
    ____
    return ____, ____


# TODO: Decompose delta_W at ranks [1, 4, 8, 16, 32, 64], print relative error
for rank in [1, 4, 8, 16, 32, 64]:
    B, A = ____
    reconstructed = ____
    relative_error = ____
    print(f"  Rank {rank:>3}: relative error={relative_error:.6f}, params={____}")

# TODO: Verify full-rank reconstruction is exact (within float tolerance)
B_full, A_full = ____
assert np.allclose(____), "Full rank should reconstruct exactly"
print("\nFull-rank reconstruction verified.")



## TASK 2: Measure reconstruction error as function of rank


In [ ]:
# TODO: Loop over ranks 1..64, collect errors, param_counts, param_savings
ranks = list(range(1, 65))
errors = []
param_counts = []
param_savings = []
full_params = ____

for r in ranks:
    B, A = ____
    recon_error = ____
    lora_params = ____
    errors.append(____)
    param_counts.append(____)
    param_savings.append(____)

# TODO: SVD spectrum analysis — cumulative energy distribution
# Hint: np.linalg.svd(delta_W, compute_uv=False) -> singular values
#   cumulative_energy = np.cumsum(S**2) / np.sum(S**2)
____
cumulative_energy = ____

print(f"\n=== Singular Value Energy Distribution ===")
for pct in [0.90, 0.95, 0.99, 0.999]:
    # TODO: Use np.searchsorted to find rank for this % energy
    rank_needed = ____
    savings = ____
    print(
        f"  {pct*100:.1f}% energy at rank {rank_needed:>3} ({savings*100:.1f}% savings)"
    )

# TODO: Build summary DataFrame with ranks [4, 8, 16, 32, 64]
results_df = pl.DataFrame(
    {
        "rank": ____,
        "relative_error": ____,
        "param_count": ____,
        "param_savings_pct": ____,
    }
)
print(f"\n=== Rank vs Error vs Parameters ===")
print(results_df)



## TASK 3: Implement LoRA forward pass from scratch


In [ ]:
# LoRA: h = W_0 @ x + (B @ A) @ x * (alpha / r)
# During training, only B and A are updated; W_0 is frozen.


class LoRALinear:
    """LoRA-augmented linear layer (from-scratch implementation).

    Original: h = W @ x
    LoRA:     h = W @ x + (B @ A) @ x * (alpha / r)
    """

    def __init__(self, W: np.ndarray, rank: int, alpha: float = 1.0):
        # TODO: Store frozen W copy, rank, alpha
        self.W = ____
        self.rank = ____
        self.alpha = ____
        d_out, d_in = W.shape
        # TODO: B=zeros(d_out, rank) — delta=0 at init; A ~ N(0, sqrt(2/d_in))
        self.B = ____
        self.A = ____
        self.scaling = ____

    def forward(self, x: np.ndarray) -> np.ndarray:
        """Forward pass: h = W @ x + B @ A @ x * (alpha / r)."""
        # TODO: h_base + (B @ A) @ x * scaling
        ____

    @property
    def trainable_params(self) -> int:
        return ____

    @property
    def total_params(self) -> int:
        return ____

    def merge(self) -> np.ndarray:
        """Merge LoRA weights: W' = W + B @ A * scaling."""
        return ____


# TODO: Create LoRALinear(W_pretrained, rank=16, alpha=32.0), test input x
x = ____
lora_layer = ____

# TODO: Verify at init h_base == h_lora (B=0)
h_base = ____
h_lora = ____
assert np.allclose(h_base, h_lora, atol=1e-6), "At init, LoRA should not change output"

print(f"\n=== LoRA Layer (rank=16, alpha=32) ===")
print(f"  Trainable: {lora_layer.trainable_params:,} / {lora_layer.total_params:,}")
print(
    f"  Trainable fraction: {lora_layer.trainable_params / lora_layer.total_params:.4%}"
)

# TODO: Simulate training — set B to small random, verify output changes
lora_layer.B = ____
h_after = ____
print(f"  After B update: max|delta| = {____:.4f}")

# TODO: Verify merge: W_merged @ x should equal forward(x)
W_merged = ____
h_merged = ____
assert np.allclose(h_after, h_merged, atol=1e-5), "Merged should match forward"
print(f"  Merge verified.")



## TASK 4: Use AlignmentPipeline with LoRAConfig at ranks 4, 16, 64


In [ ]:
rank_configs = {}

for rank in [4, 16, 64]:
    # TODO: alpha = rank * 2; create AlignmentConfig with method="sft",
    #   LoRAConfig(rank, alpha, dropout=0.05, target_modules=("q_proj","v_proj")),
    #   SFTConfig(epochs=3, batch=4, lr=2e-4, max_seq=512)
    alpha = ____
    config = AlignmentConfig(
        method=____,
        base_model_id=____,
        lora=LoRAConfig(____),
        sft=SFTConfig(____),
        experiment_dir=____,
    )
    rank_configs[rank] = config

    # TODO: Estimate LoRA params: n_layers * n_modules * 2 * d_model * rank
    lora_params = ____
    total_model_params = 1_100_000_000
    print(
        f"  Rank {rank}: {lora_params:,} params ({lora_params/total_model_params:.4%})"
    )

print(f"\n=== AlignmentPipeline API Demo ===")
registry = AdapterRegistry()
for rank, config in rank_configs.items():
    pipeline = ____
    print(f"  Pipeline: method={config.method}, rank={config.lora.rank}")

print(f"\nProduction: result = await pipeline.train(dataset=..., adapter_name='...')")



## TASK 5: Implement rank selector heuristic (stable rank)


In [ ]:
# Stable rank: r_stable(W) = ||W||_F^2 / ||W||_2^2


def stable_rank(W: np.ndarray) -> float:
    """stable_rank(W) = sum(sigma_i^2) / sigma_max^2"""
    # TODO: SVD singular values, compute fro_sq / spectral_sq
    ____
    ____
    ____
    return ____


def recommend_lora_rank(W: np.ndarray, budget_fraction: float = 0.01) -> dict:
    """Recommend LoRA rank: stable rank as upper bound, budget constraint, power-of-2."""
    d, k = W.shape
    sr = ____
    # TODO: max_rank_budget = int(budget_fraction * d * k / (d + k))
    max_rank_budget = ____
    # TODO: rank_from_stable = max(1, int(sr * 0.5))
    rank_from_stable = ____
    # TODO: min of budget and stable, then clip to power of 2
    recommended = ____
    recommended = ____

    return {
        "stable_rank": sr,
        "rank_from_stable": rank_from_stable,
        "max_rank_budget": max_rank_budget,
        "recommended_rank": recommended,
        "dimensions": (d, k),
    }


# TODO: Test on 3 matrix types: random (512x512), low-rank (intrinsic=8), near-identity
print(f"\n=== Rank Selection Heuristic ===")
W_random = ____
rec_random = ____
print(
    f"  Random: stable_rank={rec_random['stable_rank']:.1f}, recommended={rec_random['recommended_rank']}"
)

W_lowrank = ____
rec_lowrank = ____
print(
    f"  Low-rank: stable_rank={rec_lowrank['stable_rank']:.1f}, recommended={rec_lowrank['recommended_rank']}"
)

W_identity = ____
rec_identity = ____
print(
    f"  Identity: stable_rank={rec_identity['stable_rank']:.1f}, recommended={rec_identity['recommended_rank']}"
)

print(f"\n  Key insight: stable rank measures intrinsic dimensionality.")
print(f"  Low stable rank -> small LoRA rank suffices. Default: rank=16 for 7B models.")

print("\n=== Exercise 1 complete ===")

